# Hướng dẫn chạy Đánh Giá Qwen bằng vLLM (Fix lỗi Kaggle T4)

Bởi vì Kaggle cho bạn tới tận **2 card đồ họa NVIDIA T4 (16GB VRAM mỗi card)**, việc dùng **vLLM** là một quyết định tuyệt vời để đánh giá song song.

Trong Notebook này, chúng ta sẽ tải trực tiếp file GGUF từ Repo Hugging Face của bạn xuống và nạp vào vLLM!

**Bước 1:** Trong thanh menu bên phải, phần **Accelerator**, hãy chọn **GPU T4 x2**.
**Bước 2:** Upload file `dataset.jsonl` của bạn vào Kaggle (mục Input bên phải).
**Bước 3:** Vào **Add-ons -> Secrets** và cấu hình `OPENAI_API_KEY`.
**Bước 4:** Bấm "Run All".

### 1. Cài đặt thư viện (vLLM, Xformers, OpenAI...)
**Lưu ý:** Thêm `xformers` để sửa triệt để lỗi biên dịch Ninja C++ trên Kaggle.

In [ ]:
# [QUAN TRỌNG] Sửa lỗi vLLM không tìm thấy file libcuda.so khi biên dịch trên Kaggle
import os
os.system('find /usr/lib /usr/local -name "libcuda.so.*" | head -n 1 | xargs -I {} ln -sf {} /usr/lib/x86_64-linux-gnu/libcuda.so')
os.system('find /usr/lib /usr/local -name "libcuda.so.*" | head -n 1 | xargs -I {} ln -sf {} /usr/local/cuda/lib64/libcuda.so')


In [ ]:
!pip install vllm==0.6.4 transformers==4.45.2 protobuf==3.20.3 xformers openai python-dotenv tqdm huggingface_hub "numpy<2.0.0"

### 2. Tải File GGUF từ Repo Hugging Face của bạn
Hãy **SỬA LẠI** tên Repo của bạn ở biến `HF_REPO_ID` bên dưới nhé!

In [ ]:
from huggingface_hub import hf_hub_download
import os

# -----------------------------------------------------
# BẠN HÃY SỬA TÊN REPO CỦA BẠN VÀO ĐÂY:
HF_REPO_ID = "TenCuaBan/Ten-Repo-Model"
# Tên file gguf mà bạn đã push lên (thường là unsloth.Q4_K_M.gguf)
GGUF_FILENAME = "unsloth.Q4_K_M.gguf" 
# -----------------------------------------------------

print(f"Đang tải file {GGUF_FILENAME} từ repo {HF_REPO_ID}...")
model_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=GGUF_FILENAME,
    local_dir="/kaggle/working"
)
print(f"Tải thành công! File lưu tại: {model_path}")

### 3. Nạp Model vào vLLM (Đã fix lỗi sập Kaggle)

In [ ]:
import os
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# Tải file GGUF vào vLLM trên 1 card T4 (tiết kiệm tài nguyên và không bị lỗi P2P)
import torchao
import transformers.models.qwen2
llm = LLM(
    model=model_path, 
    tokenizer="Qwen/Qwen2.5-7B-Instruct",
    tensor_parallel_size=1, 
    max_model_len=4096,
    enforce_eager=True,
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")


### 4. Script Đánh Giá Song Song (Batch Evaluation)


In [ ]:
import json
import random
import time
from collections import defaultdict
from tqdm.auto import tqdm
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# Lấy API Key từ Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")
except Exception as e:
    print("Vui lòng thiết lập OPENAI_API_KEY trong menu Add-ons -> Secrets")
    OPENAI_API_KEY = ""

# TÌM KIẾM FILE DATASET
DATASET_PATH = ""
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".jsonl"):
            DATASET_PATH = os.path.join(root, file)
            break

if not DATASET_PATH:
    DATASET_PATH = "/kaggle/working/dataset.jsonl" # fallback
REPORT_PATH   = "eval_report.json"
NUM_SAMPLES   = 50

FAITHFULNESS_PROMPT = """
You are an expert evaluator. Compare the ASSISTANT's answer against the provided CONTEXT.
Score how faithful the answer is to the context on a scale of 1 to 5.
1: Completely hallucinated or contradicts context.
3: Partially faithful, but includes hallucinated details.
5: Entirely faithful to the context.
Only output the integer score.
"""

RELEVANCE_PROMPT = """
You are an expert evaluator. Compare the ASSISTANT's answer against the provided QUESTION.
Score how relevant and helpful the answer is to the question on a scale of 1 to 5.
1: Completely irrelevant or doesn't answer the question.
3: Partially answers the question, or contains irrelevant tangents.
5: Directly and fully answers the question.
Only output the integer score.
"""

def call_openai_judge(prompt: str, retries: int = 3) -> int:
    client = OpenAI(api_key=OPENAI_API_KEY)
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=5,
            )
            text = response.choices[0].message.content.strip()
            digits = [int(s) for s in text.split() if s.isdigit()]
            return digits[0] if digits else 0
        except Exception as e:
            wait = 2 ** attempt
            print(f"  [Judge error] {e} — retrying in {wait}s")
            time.sleep(wait)
    return 0

def stratified_sample(records: list, n: int, seed: int = 42) -> list:
    by_type = defaultdict(list)
    for r in records:
        by_type[r["question_type"]].append(r)
    
    for qtype in by_type:
        random.Random(seed).shuffle(by_type[qtype])
        
    result = []
    proportions = {"insufficient": 0.1, "noisy_positive": 0.3, "positive": 0.4, "multi_chunk": 0.2}
    
    for qtype, prop in proportions.items():
        count = int(n * prop)
    result.extend(by_type.get(qtype, [])[:count])
        
    while len(result) < n:
        for qtype in by_type:
            if len(result) < n and len(by_type[qtype]) > int(n * proportions.get(qtype, 0)):
                result.append(by_type[qtype].pop())
                
    random.Random(seed).shuffle(result)
    return result[:n]

def main():
    if not OPENAI_API_KEY:
        print("Dừng: Thiếu OPENAI_API_KEY! Hãy thêm trong Kaggle Secrets.")
        return

    if not os.path.exists(DATASET_PATH):
        print(f"Dừng: Không tìm thấy file {DATASET_PATH}. Vui lòng upload file jsonl lên Kaggle.")
        return

    records = []
    with open(DATASET_PATH, encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    test_samples = stratified_sample(records, NUM_SAMPLES)
    print(f"\nStarting evaluation for {len(test_samples)} samples...")

    # --- 1. SỬ DỤNG vLLM ĐỂ SINH CÂU TRẢ LỜI SONG SONG CHO TOÀN BỘ 50 CÂU ---
    print("\n[1/2] Generating answers with vLLM (Super Fast)...")
    prompts = []
    for sample in test_samples:
        messages = sample['messages'][:-1] # Lấy system và user prompt
        # Áp dụng chuẩn Chat Template của Qwen2.5
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)

    # Gửi cả 50 câu vào vLLM một lúc (Batching)
    sampling_params = SamplingParams(temperature=0.1, max_tokens=1024)
    # CHỐNG LỖI BATCHING CỦA vLLM BẰNG CÁCH CHIA NHỎ RA TỪNG CỤM 5 CÂU MỘT
    outputs = []
    for i in tqdm(range(0, len(prompts), 5), desc="Generating Batches"):
        batch_prompts = prompts[i:i+5]
        batch_outputs = llm.generate(batch_prompts, sampling_params, use_tqdm=False)
        outputs.extend(batch_outputs)

    # Gắn câu trả lời về lại mẫu dataset
    for i, out in enumerate(outputs):
        test_samples[i]["generated_answer"] = out.outputs[0].text.strip()
    print("✓ All answers generated successfully!")

    # --- 2. DÙNG GPT-4o-mini CHẤM ĐIỂM (Cần chạy vòng lặp vì gọi API qua mạng) ---
    print("\n[2/2] Evaluating answers with GPT-4o-mini...")
    results = []
    total_faithfulness, total_relevance = 0, 0
    refusal_correct, refusal_total = 0, 0
    subset_counts = {"insufficient": 0, "noisy_positive": 0, "positive": 0, "multi_chunk": 0}

    for sample in tqdm(test_samples, desc="Judging"):
        q_type = sample["question_type"]
        user_content = sample['messages'][1]['content']
        context = user_content.split('[Question]')[0].replace('[Context]', '').strip()
        question = user_content.split('[Question]')[-1].strip()
        answer = sample["generated_answer"]

        faith_prompt = f"{FAITHFULNESS_PROMPT}\n\nCONTEXT:\n{context}\n\nASSISTANT's ANSWER:\n{answer}"
        faith_score = call_openai_judge(faith_prompt)

        rel_prompt = f"{RELEVANCE_PROMPT}\n\nQUESTION:\n{question}\n\nASSISTANT's ANSWER:\n{answer}"
        rel_score = call_openai_judge(rel_prompt)

        is_refusal = any(phrase in answer.lower() for phrase in ["không đủ thông tin", "không có thông tin", "insufficient"])
        ref_corr = (q_type == "insufficient" and is_refusal)

        sample_record = {
            "question_id": sample.get("question_id", ""),
            "question": question,
            "question_type": q_type,
            "expected_answer": sample.get("answer", ""),
            "generated_answer": answer,
            "faithfulness": faith_score,
            "relevance": rel_score,
            "refusal_correct": ref_corr
        }
        results.append(sample_record)

        total_faithfulness += faith_score
        total_relevance += rel_score
        if q_type == "insufficient":
            refusal_total += 1
            if ref_corr:
                refusal_correct += 1

    avg_faithfulness = total_faithfulness / NUM_SAMPLES
    avg_relevance = total_relevance / NUM_SAMPLES
    refusal_rate = (refusal_correct / refusal_total * 100) if refusal_total > 0 else 0.0

    print("\n" + "="*40)
    print("       FINAL EVALUATION REPORT")
    print("="*40)
    print(f"Total Samples Evaluated : {NUM_SAMPLES}")
    print(f"Average Faithfulness    : {avg_faithfulness:.2f} / 5.0")
    print(f"Average Relevance       : {avg_relevance:.2f} / 5.0")
    print(f"Refusal Accuracy        : {refusal_rate:.1f}% ({refusal_correct}/{refusal_total})")
    print("="*40)
    
    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Report saved to: {REPORT_PATH}")

main()
